# Rule Extraction Engine — prototype

> **PROTOTYPE — throwaway code.** This notebook answers one question and is not part of the app.

**Question:** given the IU matriculation handbook and a free-text program description, can an LLM pipeline author admission rules that (1) compile through the real `app/rules_engine` compiler and (2) reproduce the hand-authored policy's decisions on the 16 saved gold applicants?

**Architecture** (each stage is a cell; every intermediate is printed and saved to `artifacts/`):

```
handbook.md ──> [0 ToC index] ──> [1 chunkless retrieval]        (agentic navigation, no chunks/embeddings)
                                        │ opened sections
program description ──────────────────> [2 requirements extraction]   (structured, with citations)
                                        │ requirements.json
engine vocabulary (introspected) ─────> [3 vocabulary mapping]        (supported vs UNSUPPORTED report)
                                        │ mapping.json
                            ┌─────────> [4 YAML generation — three arms]
  rules/README.md (DSL spec) ──────┤             Arm A: spec only
  hand-authored YAML ───────┤             Arm B: spec + few-shot (copy-detection control)
  structure-only skeleton ──┘             Arm C: spec + envelope skeleton, no policy content
                                        │ generated-rules-{a,b}/
                                        [5 compile + bounded repair]  (RulesEngine.activate is the guardrail)
                                        │ activated engines
                                        [6 gold evaluation]           (16 saved applicants vs hand-authored baseline)
```

**Honesty note:** `rules/README.md` (the DSL spec, which every arm must see) embeds the hand-authored `GERMAN_ABITUR` rule as its worked example. The other four rules, the shared requirements, and the policy resolution are still derived from the handbook. Arm B additionally sees all hand-authored YAML — that contrast is the experiment.

In [1]:
# PROTOTYPE — throwaway demo, not production code.
import json
import os
import re
import shutil
import sys
from pathlib import Path

from pydantic import BaseModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "rules").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "rules").is_dir(), "run this notebook from the project root or any folder inside it"
sys.path.insert(0, str(PROJECT_ROOT))

RULES_DIR = PROJECT_ROOT / "rules"
HANDBOOK = PROJECT_ROOT / "case-study" / "IU-FS-LF-Leitfaden-Hochschulzugangsberechtigung-Stand-Januar2025.md"
RUNS_DIR = PROJECT_ROOT / "runs"
OUT_DIR = PROJECT_ROOT / "tools" / "rule-extractor"
ARTIFACTS_DIR = OUT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Pick up OPENAI_API_KEY from the project .env if the shell did not export it.
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip().strip('"'))

from openai import OpenAI

MODEL = os.environ.get("ADMISSIONS_OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI()


def ask(instructions: str, user_input: str, schema: type[BaseModel]) -> BaseModel:
    """One structured-output call; the schema is enforced by the API."""
    response = client.responses.parse(
        model=MODEL,
        instructions=instructions,
        input=user_input,
        text_format=schema,
        store=False,
    )
    if response.status != "completed":
        raise RuntimeError(f"model response status: {response.status}")
    return response.output_parsed


def save_artifact(name: str, payload) -> None:
    (ARTIFACTS_DIR / name).write_text(json.dumps(payload, indent=2, ensure_ascii=False))


print(f"project: {PROJECT_ROOT}")
print(f"model:   {MODEL}")

project: <repo root>
model:   gpt-5.6-terra


## Stage 0 — Ingest the handbook and build the ToC index

The handbook markdown stands in for the uploaded PDF (the English conversion already exists; PDF→text is out of demo scope). We parse the `##`/`###` heading tree into a numbered index — **this index is the only thing the retrieval stage is allowed to see up front**. The document's own structure is the retrieval index: no chunking, no embeddings.

In [2]:
assert HANDBOOK.exists(), f"{HANDBOOK}\nPut the English markdown handbook at this path. It is IU material and is not in the repository: see case-study/README.md."
handbook_lines = HANDBOOK.read_text().splitlines()

sections = []
for line_no, line in enumerate(handbook_lines):
    match = re.match(r"^(#{2,3}) (.+)$", line)
    if match:
        sections.append({
            "id": len(sections),
            "level": len(match.group(1)),
            "title": match.group(2).strip(),
            "start": line_no,
        })
for index, section in enumerate(sections):
    section["end"] = len(handbook_lines)
    for later in sections[index + 1:]:
        if later["level"] <= section["level"]:
            section["end"] = later["start"]
            break


def section_text(section_id: int) -> str:
    section = sections[section_id]
    return "\n".join(handbook_lines[section["start"]:section["end"]])


toc_listing = "\n".join(
    f"[{s['id']:>3}] {'    ' * (s['level'] - 2)}{s['title']}  ({s['end'] - s['start']} lines)"
    for s in sections
)
print(f"{len(sections)} sections indexed from {len(handbook_lines)} lines\n")
print(toc_listing)

142 sections indexed from 6313 lines

[  0] CONTENTS  (246 lines)
[  1] CHANGE TRACKING  (170 lines)
[  2] ACCESS TO THE BACHELOR STUDY PROGRAM  (417 lines)
[  3]     Studying with the (Fach-) Abitur (general or subject-restricted school-leaving qualification)  (151 lines)
[  4]     Studying without the (Fach-)Abitur (general or subject-restricted school-leaving qualification)  (264 lines)
[  5] GENERAL SPECIAL FEATURES OF FOREIGN QUALIFICATIONS  (19 lines)
[  6]     Anabin  (6 lines)
[  7]     Certificate of equivalence (Äquivalenzbescheinigung)  (11 lines)
[  8] PROOF OF LANGUAGE PROFICIENCY - GERMAN LANGUAGE SKILLS  (44 lines)
[  9]     Luxembourg  (11 lines)
[ 10] ACCESS TO THE BACHELOR'S DEGREE PROGRAM FOR APPLICANTS FROM AUSTRIA  (33 lines)
[ 11]     General higher education entrance qualification / Matura  (5 lines)
[ 12]     Austrian school-leaving qualifications with direct access  (8 lines)
[ 13]     Austrian school-leaving qualifications with subject-restricted access  (5 li

In [ ]:
# The user's free-text input: which program to extract rules for.
PROGRAM_DESCRIPTION = (
    "Bachelor's degree program at IU International University of Applied Sciences "
    "(for example B.Sc. Computer Science, distance learning). "
    "Extract the admission eligibility rules that determine whether an applicant "
    "may access a Bachelor's study program."
)
COMPILE_AND_EVALUATE = True  # False: author + save the rule files only (no compile guardrail, no gold eval)
POLICY_ID = "IU_BACHELOR_ACCESS"
STUDY_LEVEL = "BACHELOR"
POLICY_FILE = "bachelors-access.yaml"
EXPECTED_FILES = {POLICY_FILE, "school-access-rules.yaml", "professional-access-rules.yaml",
                  "common/requirements.yaml", "common/conditions.yaml"}
PACKAGE_TAG = ""      # prefix for the generated-rules-* output dirs
ARMS = ("A", "B", "C")

# For a program the engine cannot yet compile (e.g. M.Sc. Computer Science), use instead:
# PROGRAM_DESCRIPTION = ("Master's degree program (M.Sc.) Computer Science at IU, distance learning. "
#                        "Extract the admission eligibility rules for access to this Master's program.")
# COMPILE_AND_EVALUATE = False
# POLICY_ID = "IU_MASTER_ACCESS"
# STUDY_LEVEL = "MASTER"
# POLICY_FILE = "masters-access.yaml"
# EXPECTED_FILES = {POLICY_FILE, "master-access-rules.yaml", "common/requirements.yaml", "common/conditions.yaml"}
# PACKAGE_TAG = "master-"
# ARMS = ("B", "C")
print(PROGRAM_DESCRIPTION)
print(f"mode: {'compile + gold eval' if COMPILE_AND_EVALUATE else 'save files for human review only'}")


## Stage 1 — Chunkless retrieval: agentic navigation

The LLM sees only the ToC and the program description. Each turn it requests sections to open (by id), reads them, and may request more — until it declares coverage complete or the turn budget runs out. The navigation trace below is the retrieval rationale, fully inspectable.

In [4]:
MAX_NAV_TURNS = 3


class NavigationTurn(BaseModel):
    rationale: str
    open_section_ids: list[int]
    coverage_complete: bool


NAV_INSTRUCTIONS = (
    "You are the retrieval stage of an admissions rule extraction engine. "
    "You navigate a policy handbook using only its table of contents — no chunking, no embeddings. "
    "Goal: open exactly the sections needed to author machine-readable admission eligibility "
    "rules for the described study program. Each turn, request section ids to open; you will see "
    "the full text of every opened section on the next turn. Set coverage_complete=true only when "
    "the opened sections fully cover admission eligibility for the described program. "
    "Do not open sections irrelevant to eligibility (change logs, other study levels, formalities)."
)

opened: dict[int, str] = {}
navigation_trace = []
for turn in range(1, MAX_NAV_TURNS + 1):
    opened_blob = "\n\n".join(
        f"=== [{sid}] {sections[sid]['title']} ===\n{text}" for sid, text in opened.items()
    ) or "(none yet)"
    nav = ask(
        NAV_INSTRUCTIONS,
        f"PROGRAM DESCRIPTION:\n{PROGRAM_DESCRIPTION}\n\n"
        f"TABLE OF CONTENTS:\n{toc_listing}\n\n"
        f"OPENED SECTIONS SO FAR:\n{opened_blob}",
        NavigationTurn,
    )
    new_ids = [i for i in nav.open_section_ids if 0 <= i < len(sections) and i not in opened]
    for sid in new_ids:
        opened[sid] = section_text(sid)
    navigation_trace.append({
        "turn": turn,
        "rationale": nav.rationale,
        "opened": [sections[i]["title"] for i in new_ids],
        "coverage_complete": nav.coverage_complete,
    })
    print(f"— turn {turn} —")
    print(f"  rationale: {nav.rationale}")
    for sid in new_ids:
        print(f"  opened [{sid}] {sections[sid]['title']}")
    if nav.coverage_complete and opened:
        print("  coverage declared complete")
        break

retrieved_blob = "\n\n".join(
    f"=== [{sid}] {sections[sid]['title']} ===\n{text}" for sid, text in opened.items()
)
save_artifact("navigation-trace.json", navigation_trace)
print(f"\nretrieved {len(opened)} sections, {len(retrieved_blob.splitlines())} lines total")

— turn 1 —
  rationale: Open the core Bachelor-access rules and all eligibility-relevant variants: school qualification, non-school access, foreign/Austrian/Ukrainian qualifications, language requirements, entrance examination, prior compulsory de-registration, and English-language Bachelor requirements. Exclude program-specific special requirements because the example (B.Sc. Computer Science) is not listed among them, and exclude document/formality sections.
  opened [2] ACCESS TO THE BACHELOR STUDY PROGRAM
  opened [3] Studying with the (Fach-) Abitur (general or subject-restricted school-leaving qualification)
  opened [4] Studying without the (Fach-)Abitur (general or subject-restricted school-leaving qualification)
  opened [5] GENERAL SPECIAL FEATURES OF FOREIGN QUALIFICATIONS
  opened [8] PROOF OF LANGUAGE PROFICIENCY - GERMAN LANGUAGE SKILLS
  opened [10] ACCESS TO THE BACHELOR'S DEGREE PROGRAM FOR APPLICANTS FROM AUSTRIA
  opened [16] ADMISSION OF APPLICANTS FROM UKRAINE
  ope

— turn 2 —
  rationale: Opened sections cover the general Bachelor access routes (school qualifications, vocational routes, trial study and entrance examinations), foreign-qualification evaluation including Ukraine and Austria, German and English language eligibility, the foreign indirect-HZB Bachelor entrance examination, and the bar arising from compulsory de-registration. No program-specific Computer Science eligibility section exists in the contents; BA-special requirements are not applicable to the described B.Sc. Computer Science example.
  coverage declared complete

retrieved 10 sections, 1061 lines total


## Stage 2 — Requirements extraction

Per `rules/AGENTS.md`: for every requirement, capture the conditions/thresholds/exceptions, the resulting status, the information whose absence prevents a decision, and a verbatim source quote. Source requirements only — no implementation choices yet.

In [5]:
class Citation(BaseModel):
    section_title: str
    quote: str


class Requirement(BaseModel):
    requirement_id: str
    summary: str
    conditions: str
    outcome: str
    blocking_information: str
    citations: list[Citation]


class RequirementSet(BaseModel):
    requirements: list[Requirement]


EXTRACT_INSTRUCTIONS = (
    "You extract admission eligibility requirements from policy text for later conversion into "
    "machine-readable rules. For every distinct admission path or requirement in the provided "
    "sections, record: an UPPER_SNAKE_CASE requirement_id; a one-sentence summary; the exact "
    "conditions, thresholds, and exceptions; the outcome the source prescribes when met "
    "(direct access, conditional access with e.g. a trial study or entrance exam, rejection); "
    "the information whose absence or uncertainty prevents a decision; and verbatim quotes with "
    "their section titles. Extract only what the source states — do not invent policy. "
    "Cover every admission path in the sections, including ones about vocational or "
    "professional qualifications, and note requirements that apply only in special cases."
)

requirement_set = ask(
    EXTRACT_INSTRUCTIONS,
    f"PROGRAM DESCRIPTION:\n{PROGRAM_DESCRIPTION}\n\nRETRIEVED SECTIONS:\n{retrieved_blob}",
    RequirementSet,
)

requirements_json = requirement_set.model_dump()
save_artifact("requirements.json", requirements_json)

print(f"{len(requirement_set.requirements)} requirements extracted\n")
for req in requirement_set.requirements:
    print(f"• {req.requirement_id}")
    print(f"    {req.summary}")
    print(f"    conditions: {req.conditions}")
    print(f"    outcome:    {req.outcome}")
    print(f"    blocked by: {req.blocking_information}")
    for cite in req.citations[:2]:
        print(f"    source:     [{cite.section_title}] \"{cite.quote[:110]}\"")
    print()

25 requirements extracted

• HZB_CERTIFICATE_VALIDITY_FOR_THURINGIA
    A school-based higher-education entrance qualification is accepted only when its stated territorial validity covers all federal states or Thuringia, including the specified Bavaria/Saxony exception.
    conditions: Applicant proves a general higher education entrance qualification, subject-restricted higher education entrance qualification, or general/subject-restricted entrance qualification for universities of applied sciences; where the certificate says “valid for...”, it must say “valid for all federal states” or “valid for Thuringia.” A certificate stating “Fulfils the requirements for recognition in all federal states with the exception of Bavaria and Saxony” is accepted because it is valid for Thuringia.
    outcome:    Direct eligibility through the applicable school-leaving qualification, subject to any qualification-specific conditions below.
    blocked by: The qualification type, certificate restriction

## Stage 3 — Vocabulary mapping

The engine's authoring surface is **fixed in code**: five rule IDs, three candidate collections, a closed fact/enum vocabulary, four operators, and a reason-code catalog. We introspect it from the live code (no hardcoding), then ask the LLM to map each requirement onto it — or flag it as **unsupported**, stating the extension it would need. Silently dropping a policy requirement is the one unacceptable failure mode in admissions; the unsupported report is the honest output.

In [6]:
# Prototype: private compiler tables are the ground truth, so we import them directly.
from app.models.results import RULE_ORDER, ApplicationStatus, RuleStatus
from app.rules_engine.compiler import _APPLICATION_FACTS, _SOURCE_ALIAS, _SOURCE_FACTS
from app.rules_engine.reason_catalog import RULE_EXPLANATIONS


def fact_entry(spec) -> dict:
    return {
        "type": spec.kind.__name__,
        "allowed_values": sorted(spec.domain) if spec.domain else None,
    }


VOCAB = {
    "rule_ids": [rule.value for rule in RULE_ORDER],
    "rule_ids_note": "the policy must define exactly these five rule ids — no more, no fewer",
    "collections": {
        name: {"select_alias": _SOURCE_ALIAS[name], "facts": {k: fact_entry(v) for k, v in facts.items()}}
        for name, facts in _SOURCE_FACTS.items()
    },
    "application_facts": {k: fact_entry(v) for k, v in _APPLICATION_FACTS.items()},
    "operators": ["eq", "in", "gte", "lt", "all_of", "any_of", "ref"],
    "rule_statuses": [status.value for status in RuleStatus],
    "application_statuses": [status.value for status in ApplicationStatus],
    "reason_codes": {code: RULE_EXPLANATIONS[code] for code in sorted(RULE_EXPLANATIONS)},
    "reason_codes_note": "every reason_code in the YAML must be one of these exact keys",
}
VOCAB_TEXT = json.dumps(VOCAB, indent=1)
print(f"vocabulary introspected from the engine: {len(VOCAB['rule_ids'])} rule ids, "
      f"{sum(len(c['facts']) for c in VOCAB['collections'].values())} candidate facts, "
      f"{len(VOCAB['reason_codes'])} reason codes")

vocabulary introspected from the engine: 5 rule ids, 27 candidate facts, 29 reason codes


In [7]:
class RequirementMapping(BaseModel):
    requirement_id: str
    supported: bool
    target_rule_id: str | None
    facts_used: list[str]
    notes: str
    unsupported_reason: str | None


class MappingReport(BaseModel):
    mappings: list[RequirementMapping]


MAPPING_INSTRUCTIONS = (
    "You decide, for each extracted admission requirement, whether the deterministic rule engine "
    "can express it with its FIXED vocabulary (given below as JSON). A requirement is supported "
    "only if it can be evaluated using existing facts, operators, one of the five fixed rule ids, "
    "and existing reason codes. For supported requirements name the target rule id and the facts "
    "used. For unsupported requirements set supported=false and state precisely which engine "
    "extension would be needed (new fact, new collection, new rule id, new operator...). "
    "Never force a requirement onto facts that do not actually capture its meaning."
)

mapping_report = ask(
    MAPPING_INSTRUCTIONS,
    f"ENGINE VOCABULARY (JSON):\n{VOCAB_TEXT}\n\n"
    f"EXTRACTED REQUIREMENTS (JSON):\n{json.dumps(requirements_json, indent=1)}",
    MappingReport,
)

mapping_json = mapping_report.model_dump()
save_artifact("mapping.json", mapping_json)

supported = [m for m in mapping_report.mappings if m.supported]
unsupported = [m for m in mapping_report.mappings if not m.supported]

print(f"SUPPORTED ({len(supported)}):")
for m in supported:
    print(f"  • {m.requirement_id} -> {m.target_rule_id}  facts: {', '.join(m.facts_used)}")
print(f"\nUNSUPPORTED — engine extension needed ({len(unsupported)}):")
for m in unsupported:
    print(f"  • {m.requirement_id}: {m.unsupported_reason}")

SUPPORTED (0):

UNSUPPORTED — engine extension needed (25):
  • HZB_CERTIFICATE_VALIDITY_FOR_THURINGIA: A new generic school-HZB validity rule id (or permission for the existing school rules to be composed as one requirement) is needed, plus a qualification subtype fact distinguishing general versus subject-restricted Fachhochschulreife.
  • GENERAL_HIGHER_EDUCATION_ENTRANCE_QUALIFICATION: New school-certificate facts are needed for official-certification status and document language.
  • SUBJECT_RESTRICTED_HIGHER_EDUCATION_ENTRANCE_QUALIFICATION: New school-certificate facts are needed for official-certification status, document language, and required translation availability.
  • GENERAL_FACHHOCHSCHULREIFE: A new fact for official certification of the school-based part is needed. If the overall-certificate distinction is material independently of the two part proofs, a new overall-certificate-status fact is also needed.
  • SUBJECT_RESTRICTED_FACHHOCHSCHULREIFE: New school-qualificat

## Stage 4 — YAML generation, three arms

All arms get the DSL spec (`rules/README.md`), the introspected vocabulary, the two scaffold status files, and the stage-2/3 artifacts.

- **Arm A — spec only.** Run 1 finding: never compiled; it guessed the file *envelope* wrong (sources field names, imports nesting, evaluation block).
- **Arm B — spec + hand-authored YAML.** Run 1 finding: reproduced the reference byte-for-byte (versions aside) — kept as a copy-detection control, not evidence of derivation.
- **Arm C — spec + structure-only skeleton.** The skeleton shows every envelope field with `<...>` placeholders and zero policy content: it targets exactly Arm A's failure mode without handing over the answer key. This is the arm that can support a genuine "derived from the handbook" claim.


In [ ]:
POLICY_SKELETON = r"""
# ============ SKELETON 1: policy entry file ============
dsl_version: "1.3"

policy:
  id: <POLICY_ID>
  version: "<version-string>"

  applies_when:
    fact: application.study_level
    eq: <STUDY_LEVEL_VALUE>

  sources:
    - file: <relative-path-to-source-document>
      section: <SECTION TITLE>
      subsections:
        - <Subsection title>

  imports:
    - namespace: rule_statuses
      file: rule-statuses.yaml
    - namespace: application_statuses
      file: application-statuses.yaml
    - namespace: requirements
      file: common/requirements.yaml
    - namespace: <module_namespace>
      file: <rule-module-file.yaml>

  evaluation:
    rule_groups:
      - include: <module_namespace>.<EXPORTED_RULE_GROUP_NAME>

  resolution:
    first_match:
      - when_any_rule:
          ref: rule_statuses.<RULE_STATUS>
        application_status:
          ref: application_statuses.<APPLICATION_STATUS>
      # ...one case per resolution priority, in order...
      - when_all_applicable_rules:
          ref: rule_statuses.<RULE_STATUS>
        application_status:
          ref: application_statuses.<APPLICATION_STATUS>
      - when_no_recognized_rule: true
        application_status:
          ref: application_statuses.<APPLICATION_STATUS>

# ============ SKELETON 2: rule module file (imported by the policy) ============
dsl_version: "1.3"

module:
  id: <MODULE_ID>
  version: "<version-string>"
  imports:
    - namespace: rule_statuses
      file: rule-statuses.yaml
    - namespace: conditions
      file: common/conditions.yaml
  requires_namespaces:
    - requirements

  exports:
    <RULE_GROUP_NAME>:
      id: <RULE_GROUP_NAME>
      rules:
        # body form 1: require + result
        - id: <RULE_ID>
          select:
            from: <collection_name>
            as: <collection_alias>
            where:
              fact: <alias>.<fact_name>
              eq: <VALUE>
          applicability:            # optional
            require:
              ref: requirements.<exported_requirement_name>
            result:
              not_applicable:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>
              unknown:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>
          require:
            all_of:
              - ref: requirements.<exported_requirement_name>
              - fact: <alias>.<fact_name>
                eq: <VALUE>
          result:
            satisfied:
              status:
                ref: rule_statuses.<RULE_STATUS>
              reason_code: <REASON_CODE>
            not_satisfied:
              status:
                ref: rule_statuses.<RULE_STATUS>
              reason_code: <REASON_CODE>
            unknown:
              status:
                ref: rule_statuses.<RULE_STATUS>
              reason_code: <REASON_CODE>

        # body form 2: ordered branches
        - id: <RULE_ID>
          select:
            from: <collection_name>
            as: <collection_alias>
            where:
              fact: <alias>.<fact_name>
              eq: <VALUE>
          branches:
            first_match:
              - when:
                  all_of:
                    - fact: <alias>.<fact_name>
                      eq: <VALUE>
                    - ref: requirements.<exported_requirement_name>
                result:
                  status:
                    ref: rule_statuses.<RULE_STATUS>
                  reason_code: <REASON_CODE>
                  condition: conditions.<CONDITION_NAME>   # only on conditional results
            unknown:
              result:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>
            otherwise:
              result:
                status:
                  ref: rule_statuses.<RULE_STATUS>
                reason_code: <REASON_CODE>

# ============ SKELETON 3: shared definitions module (common/requirements.yaml and common/conditions.yaml) ============
dsl_version: "1.3"

module:
  id: <MODULE_ID>
  version: "<version-string>"

  exports:
    # in common/requirements.yaml: exported names are lowercase expressions
    <exported_requirement_name>:
      fact: <alias>.<fact_name>
      eq: <VALUE>
    <another_requirement_name>:
      any_of:
        - fact: <alias>.<fact_name>
          eq: <VALUE>
        - fact: <alias>.<fact_name>
          in:
            - <VALUE>
            - <VALUE>
    # in common/conditions.yaml: exported names are UPPERCASE, free-form parameter mappings
    # <CONDITION_NAME>:
    #   <parameter>: <value>
"""


class GeneratedFile(BaseModel):
    path: str
    content: str


class RulePackage(BaseModel):
    files: list[GeneratedFile]
    proposed_extensions: list[str] = []


SPEC_TEXT = (PROJECT_ROOT / "rules" / "README.md").read_text()
SCAFFOLD_FILES = {
    name: (RULES_DIR / name).read_text()
    for name in ("rule-statuses.yaml", "application-statuses.yaml")
}
FEWSHOT_FILES = {
    name: (RULES_DIR / name).read_text()
    for name in (
        "bachelors-access.yaml",
        "school-access-rules.yaml",
        "professional-access-rules.yaml",
        "common/requirements.yaml",
        "common/conditions.yaml",
    )
}


def build_instructions(arm: str) -> str:
    scaffold_blob = "\n\n".join(f"--- {name} (provided verbatim) ---\n{text}" for name, text in SCAFFOLD_FILES.items())
    if COMPILE_AND_EVALUATE:
        role = ("You are the rule authoring stage of an admissions rule extraction engine. From the "
                "extracted requirements and their vocabulary mapping, author a complete DSL 1.3 rule "
                "package that the deterministic compiler accepts.")
        contract = (
            "HARD REQUIREMENTS:\n"
            "- Produce exactly these files: " + ", ".join(sorted(EXPECTED_FILES)) + "\n"
            f"- The policy file is {POLICY_FILE} with policy id {POLICY_ID}, "
            f"applies_when application.study_level eq {STUDY_LEVEL}, and version \"0.1.0-extracted\".\n"
            "- The policy must define exactly the five rule ids from the vocabulary — no more, no fewer.\n"
            "- Every reason_code must be an exact key from the vocabulary's reason_codes.\n"
            "- Only use facts, values, and operators from the vocabulary; proposed_extensions must stay empty.\n"
            "- Every branch group needs unknown and otherwise results; requirements need satisfied, "
            "not_satisfied, and unknown results.\n"
            "- The policy resolution first_match must end with a when_no_recognized_rule case.\n"
            "- Record the handbook file and section titles in the policy sources field.\n"
            "- Unsupported requirements are out of scope: encode only supported ones."
        )
    else:
        role = ("You are the rule authoring stage of an admissions rule extraction engine. From the "
                "extracted requirements and their vocabulary mapping, author a PROPOSED DSL 1.3 rule "
                "package for the described program. The current engine vocabulary does not cover this "
                "program, so the package is for HUMAN REVIEW, not compilation.")
        contract = (
            "REQUIREMENTS FOR THE PROPOSED PACKAGE:\n"
            "- Produce exactly these files: " + ", ".join(sorted(EXPECTED_FILES)) + "\n"
            f"- The policy file is {POLICY_FILE} with policy id {POLICY_ID}, "
            f"applies_when application.study_level eq {STUDY_LEVEL}, and version \"0.1.0-proposed\".\n"
            "- You MAY propose new rule ids, facts, collections, enum values, and reason codes where "
            "this program needs them — follow the naming style of the existing vocabulary.\n"
            "- Every proposed vocabulary item that does not exist in the current engine must appear in "
            "proposed_extensions as one line each: '<kind>: <name> — <why needed>' (kinds: rule_id, "
            "collection, fact, enum_value, reason_code, condition, operator).\n"
            "- Keep every structural DSL rule: explicit satisfied/not_satisfied/unknown results, "
            "unknown and otherwise in every branch group, resolution first_match ending with "
            "when_no_recognized_rule, sources recorded with file/section/subsections.\n"
            "- Encode only what the handbook states; unclear or conflicting source text becomes "
            "MANUAL_REVIEW outcomes, not invented policy."
        )
    parts = [
        role,
        f"THE DSL SPECIFICATION:\n{SPEC_TEXT}",
        f"THE ENGINE VOCABULARY (JSON):\n{VOCAB_TEXT}",
        f"SCAFFOLD FILES ALREADY PRESENT IN THE PACKAGE — import them, never regenerate them:\n{scaffold_blob}",
        contract,
    ]
    if arm == "B":
        fewshot_blob = "\n\n".join(f"--- {name} ---\n{text}" for name, text in FEWSHOT_FILES.items())
        label = ("hand-authored files for the same policy" if COMPILE_AND_EVALUATE
                 else "hand-authored BACHELOR files; a different program")
        parts.append(f"REFERENCE IMPLEMENTATION ({label}; use as structure and style examples):\n{fewshot_blob}")
    if arm == "C":
        parts.append(
            "FILE STRUCTURE SKELETON — structure only. Every <...> placeholder must be replaced "
            "using the vocabulary, the DSL spec, and the extracted requirements; the skeleton "
            f"carries no policy content:\n{POLICY_SKELETON}"
        )
    return "\n\n".join(parts)


def generate_package(arm: str, error_feedback: str | None = None, previous: RulePackage | None = None) -> RulePackage:
    parts = [
        f"PROGRAM DESCRIPTION:\n{PROGRAM_DESCRIPTION}",
        f"EXTRACTED REQUIREMENTS (JSON):\n{json.dumps(requirements_json, indent=1)}",
        f"VOCABULARY MAPPING (JSON):\n{json.dumps(mapping_json, indent=1)}",
    ]
    if error_feedback is not None and previous is not None:
        previous_blob = "\n\n".join(f"--- {f.path} ---\n{f.content}" for f in previous.files)
        parts += [
            f"YOUR PREVIOUS ATTEMPT:\n{previous_blob}",
            f"THE COMPILER REJECTED IT WITH:\n{error_feedback}\n\nReturn the full corrected file set.",
        ]
    return ask(build_instructions(arm), "\n\n".join(parts), RulePackage)


def write_package(arm_dir: Path, package: RulePackage) -> None:
    if arm_dir.exists():
        shutil.rmtree(arm_dir)
    (arm_dir / "common").mkdir(parents=True)
    for name, text in SCAFFOLD_FILES.items():
        (arm_dir / name).write_text(text)
    for file in package.files:
        if file.path not in EXPECTED_FILES:
            print(f"    skipping unexpected file: {file.path}")
            continue
        (arm_dir / file.path).write_text(file.content)


print(f"generation helpers ready; arms: {', '.join(ARMS)}; "
      f"mode: {'strict compile contract' if COMPILE_AND_EVALUATE else 'proposed package for review'}")


## Stage 5 — Compile with bounded repair

`RulesEngine.activate()` runs the real compiler: unknown facts, off-catalog reason codes, a wrong rule-id set, or a malformed resolution all reject the package with a typed error. On rejection, the error goes back to the model — at most 3 attempts per arm. The repair transcript is part of the result.

In [ ]:
from app.rules_engine import RulesEngine

MAX_COMPILE_ATTEMPTS = 3


def compile_with_repair(arm: str) -> dict:
    arm_dir = OUT_DIR / f"generated-rules-{PACKAGE_TAG}{arm.lower()}"
    print(f"=== Arm {arm} ===")
    package = generate_package(arm)
    for attempt in range(1, MAX_COMPILE_ATTEMPTS + 1):
        write_package(arm_dir, package)
        try:
            engine = RulesEngine.activate(arm_dir)
            print(f"  attempt {attempt}: COMPILED")
            return {"arm": arm, "dir": arm_dir, "engine": engine, "attempts": attempt, "error": None}
        except Exception as error:  # PolicyActivationError or spec validation errors
            code = getattr(error, "code", type(error).__name__)
            message = getattr(error, "safe_message", str(error))
            print(f"  attempt {attempt}: REJECTED [{code}] {message}")
            if attempt == MAX_COMPILE_ATTEMPTS:
                return {"arm": arm, "dir": arm_dir, "engine": None, "attempts": attempt,
                        "error": f"{code}: {message}"}
            package = generate_package(arm, error_feedback=f"{code}: {message}", previous=package)


def save_only(arm: str) -> dict:
    arm_dir = OUT_DIR / f"generated-rules-{PACKAGE_TAG}{arm.lower()}"
    print(f"=== Arm {arm} (save only) ===")
    package = generate_package(arm)
    write_package(arm_dir, package)
    save_artifact(f"proposed-extensions-{PACKAGE_TAG}{arm.lower()}.json", package.proposed_extensions)
    print(f"  wrote {len(package.files)} files -> {arm_dir.name}/")
    print(f"  {len(package.proposed_extensions)} proposed engine extensions")
    try:  # documented compile attempt, no repair — rejection expected and informative
        RulesEngine.activate(arm_dir)
        print("  note: package unexpectedly compiles against the current engine")
    except Exception as error:
        code = getattr(error, "code", type(error).__name__)
        message = getattr(error, "safe_message", str(error))
        print(f"  compile attempt (expected rejection): [{code}] {message}")
    return {"arm": arm, "dir": arm_dir, "engine": None, "attempts": 1, "error": None}


runner = compile_with_repair if COMPILE_AND_EVALUATE else save_only
arms = {arm: runner(arm) for arm in ARMS}
if COMPILE_AND_EVALUATE:
    for arm, outcome in arms.items():
        verdict = "compiled" if outcome["engine"] else f"FAILED ({outcome['error']})"
        print(f"Arm {arm}: {verdict} after {outcome['attempts']} attempt(s) -> {outcome['dir'].name}/")


## Stage 6 — Gold evaluation: 16 saved applicants

Baseline = the **current hand-authored rules** evaluated on the same artifacts (the saved `decision-report.json` files were produced by policy v0.0.19 and are shown as a secondary reference). The evaluator pins the policy version, so each artifact's `program.policy.version` is patched in memory to match the engine being tested — nothing on disk changes. Agreement is measured on the final application status and on all five per-rule statuses.

In [ ]:
if not COMPILE_AND_EVALUATE:
    print("skipped — no compile/eval in save-only mode; review the generated files and "
          "the proposed-extensions artifact instead")
else:
    from ruamel.yaml import YAML

    from app.io.artifact_io import load_facts_artifact
    from app.models.outcomes import EvaluationSucceeded


    def policy_version(rules_dir: Path) -> str:
        document = YAML(typ="safe").load((rules_dir / "bachelors-access.yaml").read_text())
        return str(document["policy"]["version"])


    def with_policy_version(artifact, version: str):
        policy = artifact.program.policy.model_copy(update={"version": version})
        program = artifact.program.model_copy(update={"policy": policy})
        return artifact.model_copy(update={"program": program})


    gold_runs = sorted(d for d in RUNS_DIR.iterdir() if (d / "application-facts.json").exists())
    artifacts = {d.name: load_facts_artifact(d / "application-facts.json") for d in gold_runs}
    saved_reports = {
        d.name: json.loads((d / "decision-report.json").read_text()).get("application_status", "—")
        for d in gold_runs
    }


    def decide_all(engine: RulesEngine, version: str) -> dict[str, dict]:
        decisions = {}
        for name, artifact in artifacts.items():
            outcome = engine.evaluate(with_policy_version(artifact, version))
            if isinstance(outcome, EvaluationSucceeded):
                result = outcome.result
                decisions[name] = {
                    "status": result.application_status.value,
                    "rules": {rule.rule_id.value: rule.status.value for rule in result.rules},
                }
            else:
                decisions[name] = {"status": f"FAILED:{outcome.failure.code}", "rules": {}}
        return decisions


    baseline = decide_all(RulesEngine.activate(RULES_DIR), policy_version(RULES_DIR))
    arm_decisions = {
        arm: decide_all(outcome["engine"], policy_version(outcome["dir"]))
        for arm, outcome in arms.items() if outcome["engine"]
    }

    width = max(len(name) for name in artifacts)
    header = f"{'applicant':<{width}}  {'saved(v0.0.19)':<16} {'baseline':<22}"
    for arm in arm_decisions:
        header += f" {'arm ' + arm:<22}"
    print(header)
    print("-" * len(header))
    for name in artifacts:
        row = f"{name:<{width}}  {saved_reports[name]:<16} {baseline[name]['status']:<22}"
        for arm, decisions in arm_decisions.items():
            status = decisions[name]["status"]
            marker = "=" if status == baseline[name]["status"] else "≠"
            row += f" {marker + ' ' + status:<22}"
        print(row)

In [ ]:
if not COMPILE_AND_EVALUATE:
    print("skipped — no compile/eval in save-only mode; review the generated files and "
          "the proposed-extensions artifact instead")
else:
    summary = {"model": MODEL, "program_description": PROGRAM_DESCRIPTION,
               "sections_retrieved": len(opened),
               "requirements_extracted": len(requirement_set.requirements),
               "requirements_supported": len(supported),
               "requirements_unsupported": len(unsupported),
               "arms": {}}

    total = len(artifacts)
    print(f"{'':<8}{'compiled':<10}{'attempts':<10}{'status agreement':<20}{'per-rule agreement'}")
    for arm, outcome in arms.items():
        if outcome["engine"]:
            decisions = arm_decisions[arm]
            status_hits = sum(decisions[n]["status"] == baseline[n]["status"] for n in artifacts)
            rule_hits = sum(decisions[n]["rules"] == baseline[n]["rules"] for n in artifacts)
            print(f"Arm {arm:<4}{'yes':<10}{outcome['attempts']:<10}"
                  f"{f'{status_hits}/{total}':<20}{f'{rule_hits}/{total}'}")
            summary["arms"][arm] = {"compiled": True, "attempts": outcome["attempts"],
                                     "status_agreement": f"{status_hits}/{total}",
                                     "per_rule_agreement": f"{rule_hits}/{total}",
                                     "decisions": decisions}
        else:
            print(f"Arm {arm:<4}{'NO':<10}{outcome['attempts']:<10}{'—':<20}—")
            summary["arms"][arm] = {"compiled": False, "attempts": outcome["attempts"],
                                     "error": outcome["error"]}

    summary["baseline"] = baseline
    save_artifact("summary.json", summary)
    print(f"\nartifacts saved to {ARTIFACTS_DIR}")

## Findings (run 2, 2026-08-25, model gpt-5.6-terra, three arms)

| Arm | Prompt extras | Compiled | Status agreement | Per-rule agreement |
|---|---|---|---|---|
| A | none (spec only) | no — 3/3 `INVALID_POLICY_SCHEMA` | — | — |
| B | hand-authored YAML | attempt 1 | 16/16 | 16/16 |
| C | structure-only skeleton | attempt 2 | **15/16** | **15/16** |

**Arm C is the headline.** It never saw the hand-authored rules, its YAML differs on 30–95% of lines per file, and it still reproduced the baseline decision for 15 of 16 applicants. The one repair round fixed an `INVALID_REFERENCE_TARGET` (a requirement ref pointing at a non-expression) — the typed compiler error was specific enough to steer, unlike Arm A's opaque schema error. This supports the genuine claim: *derived from the handbook, compiled by the real engine, behaviorally equivalent on the gold set except one case.*

**The one disagreement (tobias-renner)** is a semantics choice, not a syntax slip: the applicant's vocational case has `duration_months` and `full_time_equivalent_days` MISSING and `all_weekly_hours_known=false`. The hand-authored rule surfaces this as MISSING_INFORMATION (ask for documents); Arm C used the evidence-completeness booleans as hard branch requirements, so every branch evaluated FALSE and fell to `otherwise` → NOT_SATISFIED → INELIGIBLE. The handbook doesn't prescribe how incomplete evidence resolves — but for admissions, MISSING_INFORMATION is the safer policy. Exactly the kind of divergence this eval exists to catch, and an argument for keeping human review of extracted rules.

**Arm A** failed identically to run 1 (file-envelope guesses; opaque `INVALID_POLICY_SCHEMA` gives the repair loop nothing to steer by). **Arm B** again copied its reference near-verbatim — a copy-detection control, not evidence of derivation.

**Mapping stage** was even more conservative this run: 0 of 25 requirements marked fully supported (run 1: 2/26) — it now counts official-certification/document-language evidence as missing vocabulary for every route. The generator proceeded from the requirement details regardless; the unsupported report remains the honest record of what the engine vocabulary cannot express (language proof, foreign/Anabin equivalence, BW special routes, Austria/Ukraine handling, de-registration, experience arithmetic).

**Verdict:** a structure-only skeleton is enough scaffolding for genuine rule derivation — few-shot with real rules is unnecessary and epistemically harmful (it collapses into copying). Next cheap wins: richer compiler error surfacing for the repair loop, and a diff-review step for the 1-in-16 semantic divergences.

**Capture when done:** commit this folder to a throwaway branch and record the verdict — the main branch keeps only the validated decision.
